# EchoNext Secondary Label Pipeline

This notebook is for the secondary analyses after the primary composite-SHD benchmark.

Recommended labels:
- `lvef_lte_45_flag` with full lead sweep
- `rv_systolic_dysfunction_moderate_or_greater_flag` with full lead sweep
- `lvwt_gte_13_flag` with core-model seeded comparison first, then decide whether a full sweep is worth running

This notebook is parameterized so you can change the label and whether to run the full lead sweep.

## 1. Runtime setup

Recommended Colab settings:
- Runtime type: Python 3
- Hardware accelerator: L4 GPU or T4 GPU
- High-RAM: on

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/echonext_single_lead')
LOCAL_DATA_DIR = PROJECT_DIR / 'data' / 'raw'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/echonext-a-dataset-for-detecting-echocardiogram-confirmed-structural-heart-disease-from-ecgs-1.1.1')

# Change this to the secondary label you want to run.
LABEL = 'lvef_lte_45_flag'

# For lvef_lte_45_flag and rv_systolic_dysfunction_moderate_or_greater_flag,
# keep this True. For lvwt_gte_13_flag you may want to start with False.
RUN_FULL_SWEEP = True

SEEDS = '42 43 44'
EPOCHS = 5
BATCH_SIZE = 16
BOOTSTRAP_ITERATIONS = 1000
OUTPUT_DIR = f'secondary_outputs/{LABEL}'
DRIVE_RESULTS_DIR = Path(f'/content/drive/MyDrive/echonext_outputs_{LABEL}')

print('PROJECT_DIR =', PROJECT_DIR)
print('LOCAL_DATA_DIR =', LOCAL_DATA_DIR)
print('DRIVE_DATA_DIR =', DRIVE_DATA_DIR)
print('LABEL =', LABEL)
print('RUN_FULL_SWEEP =', RUN_FULL_SWEEP)
print('SEEDS =', SEEDS)
print('EPOCHS =', EPOCHS)
print('BATCH_SIZE =', BATCH_SIZE)
print('BOOTSTRAP_ITERATIONS =', BOOTSTRAP_ITERATIONS)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/echonext_single_lead
!git clone https://github.com/aavascular/echonext_single_lead.git /content/echonext_single_lead
!ls /content/echonext_single_lead


In [ ]:
!pip install -r /content/echonext_single_lead/requirements.txt


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Verify and copy required data

In [ ]:
required_files = [
    'echonext_metadata_100k.csv',
    'EchoNext_train_waveforms.npy',
    'EchoNext_val_waveforms.npy',
    'EchoNext_test_waveforms.npy',
    'EchoNext_train_tabular_features.npy',
    'EchoNext_val_tabular_features.npy',
    'EchoNext_test_tabular_features.npy',
]

missing = [name for name in required_files if not (DRIVE_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')
print('All required Drive files found.')


In [ ]:
import shutil

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in required_files:
    src = DRIVE_DATA_DIR / name
    dst = LOCAL_DATA_DIR / name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print('Skipping existing file:', name)
        continue
    print('Copying', name)
    shutil.copy2(src, dst)

print('\nLocal data directory contents:')
for path in sorted(LOCAL_DATA_DIR.glob('*')):
    print(' -', path.name)


## 3. Inspect the selected label prevalence

In [ ]:
%cd /content/echonext_single_lead
!python3 scripts/01_inspect_data.py --data_dir data/raw


## 4. Optional full lead sweep

For `lvef_lte_45_flag` and `rv_systolic_dysfunction_moderate_or_greater_flag`, keep `RUN_FULL_SWEEP = True`.
For `lvwt_gte_13_flag`, you may want to set it to `False` initially.

In [ ]:
import subprocess

if RUN_FULL_SWEEP:
    cmd = [
        'python3', 'scripts/04_run_lead_sweep.py',
        '--data_dir', 'data/raw',
        '--label', LABEL,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--output_dir', OUTPUT_DIR,
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Skipping full lead sweep for this label.')


## 5. Run repeated-seed stability for the core publication models

In [ ]:
!python3 scripts/07_run_seed_stability.py --data_dir data/raw --label {LABEL} --seeds {SEEDS} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --include_six_limb --output_dir {OUTPUT_DIR}


## 6. Build publication-style summaries and confidence intervals

In [ ]:
!python3 scripts/08_make_publication_results.py --label {LABEL} --seeds {SEEDS} --include_six_limb --bootstrap_iterations {BOOTSTRAP_ITERATIONS} --output_dir {OUTPUT_DIR}


## 7. Build aggregate benchmark tables and figures

In [ ]:
!python3 scripts/06_make_results_tables.py --output_dir {OUTPUT_DIR}


## 8. Inspect generated outputs

In [ ]:
!find {OUTPUT_DIR} -maxdepth 3 -type f | sort


In [ ]:
import pandas as pd
from pathlib import Path

output_root = Path(OUTPUT_DIR)

publication_core_df = pd.read_csv(output_root / 'tables' / 'publication_core_results.csv')
seed_stability_df = pd.read_csv(output_root / 'tables' / 'seed_stability_summary.csv')
print('Publication core results')
display(publication_core_df)

print('Seed stability summary')
display(seed_stability_df)

benchmark_path = output_root / 'tables' / 'model_performance_by_input.csv'
if benchmark_path.exists():
    print('Benchmark table')
    display(pd.read_csv(benchmark_path))
else:
    print('No lead sweep benchmark table found for this run.')


In [ ]:
from IPython.display import Image, display
from pathlib import Path

figure_paths = [
    Path(OUTPUT_DIR) / 'figures' / 'publication_core_roc.png',
    Path(OUTPUT_DIR) / 'figures' / 'publication_core_pr.png',
    Path(OUTPUT_DIR) / 'figures' / 'publication_core_auroc_ci.png',
    Path(OUTPUT_DIR) / 'figures' / 'publication_core_auprc_ci.png',
    Path(OUTPUT_DIR) / 'figures' / 'seed_stability_auroc.png',
    Path(OUTPUT_DIR) / 'figures' / 'auroc_by_input.png',
    Path(OUTPUT_DIR) / 'figures' / 'auprc_by_input.png',
    Path(OUTPUT_DIR) / 'figures' / 'calibration_plot.png',
]

for figure_path in figure_paths:
    if figure_path.exists():
        print('\n', figure_path)
        display(Image(filename=str(figure_path)))


## 9. Copy all outputs back to Google Drive

In [ ]:
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
!rm -rf {DRIVE_RESULTS_DIR}
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
!cp -r {OUTPUT_DIR} {DRIVE_RESULTS_DIR}/
!find {DRIVE_RESULTS_DIR} -maxdepth 4 -type f | sort
